Today's objectives
1. Advanced filtering with multiple conditions.
2. Selecting and modifying rows and columns with .loc.
3. Merging multiple DataFrames.
4. Understanding unmatched records after a merge.
5. Creating new columns using vectorized operations.
6. A small business analysis exercise.

In [2]:
import pandas as pd
import numpy as np

In [3]:
data = {
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006,
                1007, 1008, 1009, 1010, 1011, 1012],
    "Product": ["Laptop", "Mouse", "Keyboard", "Laptop",
                "Monitor", "Mouse", "Keyboard", "Desk",
                "Laptop", "Monitor", "Mouse", "Desk"],
    "Category": ["Electronics", "Electronics", "Electronics",
                 "Electronics", "Electronics", "Electronics",
                 "Electronics", "Furniture", "Electronics",
                 "Electronics", "Electronics", "Furniture"],
    "Quantity": [2, 5, 3, 1, np.nan, 4, 2, 1, 3, 2, np.nan, 2],
    "UnitPrice": [800, 20, 50, 800, 250, 20, 50, 300,
                  800, 250, 20, 300],
    "Region": ["North", "South", "North", "East", "West",
               "South", "North", "East", "West", "West",
               "South", "East"],
    "OrderDate": ["2026-09-01", "2026-09-02", "2026-09-03",
                  "2026-09-04", "2026-09-05", "2026-09-06",
                  "2026-09-07", "2026-09-08", "2026-09-09",
                  "2026-09-10", "2026-09-11", "2026-09-12"]
}

df = pd.DataFrame(data)
df["OrderDate"]=pd.to_datetime(df["OrderDate"])
df_no_missing=df.copy()
df_no_missing["Quantity_imputed"] = df_no_missing["Quantity"].isna()
df_no_missing["Quantity"] = df.groupby(
    ["Product", "Region"]
)["Quantity"].transform(
    lambda x: x.fillna(np.ceil(x.median()))
)
df_no_missing["Quantity"]=df_no_missing["Quantity"].astype(int)
df_no_missing["Revenue"]=df_no_missing["Quantity"]*df_no_missing["UnitPrice"]
print(df_no_missing)

    OrderID   Product     Category  Quantity  UnitPrice Region  OrderDate  \
0      1001    Laptop  Electronics         2        800  North 2026-09-01   
1      1002     Mouse  Electronics         5         20  South 2026-09-02   
2      1003  Keyboard  Electronics         3         50  North 2026-09-03   
3      1004    Laptop  Electronics         1        800   East 2026-09-04   
4      1005   Monitor  Electronics         2        250   West 2026-09-05   
5      1006     Mouse  Electronics         4         20  South 2026-09-06   
6      1007  Keyboard  Electronics         2         50  North 2026-09-07   
7      1008      Desk    Furniture         1        300   East 2026-09-08   
8      1009    Laptop  Electronics         3        800   West 2026-09-09   
9      1010   Monitor  Electronics         2        250   West 2026-09-10   
10     1011     Mouse  Electronics         5         20  South 2026-09-11   
11     1012      Desk    Furniture         2        300   East 2026-09-12   

In [4]:
df_no_missing.loc[((df_no_missing["Region"].isin(['North','West'])))
    &(df_no_missing["Revenue"]>500)
    &(df_no_missing["Quantity"]>=2),
    ["OrderID","Product","Quantity","Region","Revenue"]]

,OrderID,Product,Quantity,Region,Revenue
0,1001,Laptop,2,North,1600
8,1009,Laptop,3,West,2400


In [5]:
df_no_missing["OrderType"]=np.where(df_no_missing["Revenue"]>=1000, "High","Standard")
df_no_missing


,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate,Quantity_imputed,Revenue,OrderType
0,1001,Laptop,Electronics,2,800,North,2026-09-01,False,1600,High
1,1002,Mouse,Electronics,5,20,South,2026-09-02,False,100,Standard
2,1003,Keyboard,Electronics,3,50,North,2026-09-03,False,150,Standard
3,1004,Laptop,Electronics,1,800,East,2026-09-04,False,800,Standard
4,1005,Monitor,Electronics,2,250,West,2026-09-05,True,500,Standard
5,1006,Mouse,Electronics,4,20,South,2026-09-06,False,80,Standard
6,1007,Keyboard,Electronics,2,50,North,2026-09-07,False,100,Standard
7,1008,Desk,Furniture,1,300,East,2026-09-08,False,300,Standard
8,1009,Laptop,Electronics,3,800,West,2026-09-09,False,2400,High
9,1010,Monitor,Electronics,2,250,West,2026-09-10,False,500,Standard


In [6]:
customers = pd.DataFrame({
    "CustomerID": [201, 202, 203, 204, 205, 206],
    "CustomerName": ["Alice", "Bob", "Carol", "David", "Eva", "Frank"],
    "CustomerType": ["Regular", "Premium", "Regular", "Premium", "Regular", "Premium"]
})
df_no_missing["CustomerID"] = [
    201, 202, 203, 201, 204, 202,
    203, 205, 201, 204, 206, 205
]

In [7]:
df_merged=df_no_missing.copy()
df_merged=pd.merge(df_no_missing,customers,on="CustomerID",how="left")
df_merged

,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate,Quantity_imputed,Revenue,OrderType,CustomerID,CustomerName,CustomerType
0,1001,Laptop,Electronics,2,800,North,2026-09-01,False,1600,High,201,Alice,Regular
1,1002,Mouse,Electronics,5,20,South,2026-09-02,False,100,Standard,202,Bob,Premium
2,1003,Keyboard,Electronics,3,50,North,2026-09-03,False,150,Standard,203,Carol,Regular
3,1004,Laptop,Electronics,1,800,East,2026-09-04,False,800,Standard,201,Alice,Regular
4,1005,Monitor,Electronics,2,250,West,2026-09-05,True,500,Standard,204,David,Premium
5,1006,Mouse,Electronics,4,20,South,2026-09-06,False,80,Standard,202,Bob,Premium
6,1007,Keyboard,Electronics,2,50,North,2026-09-07,False,100,Standard,203,Carol,Regular
7,1008,Desk,Furniture,1,300,East,2026-09-08,False,300,Standard,205,Eva,Regular
8,1009,Laptop,Electronics,3,800,West,2026-09-09,False,2400,High,201,Alice,Regular
9,1010,Monitor,Electronics,2,250,West,2026-09-10,False,500,Standard,204,David,Premium


In [8]:
print(df_merged.shape)
df_merged["CustomerName"].isna().sum()

(12, 13)


0

In [9]:
df_merged[["OrderID", "CustomerID", "CustomerName", "CustomerType"]]

,OrderID,CustomerID,CustomerName,CustomerType
0,1001,201,Alice,Regular
1,1002,202,Bob,Premium
2,1003,203,Carol,Regular
3,1004,201,Alice,Regular
4,1005,204,David,Premium
5,1006,202,Bob,Premium
6,1007,203,Carol,Regular
7,1008,205,Eva,Regular
8,1009,201,Alice,Regular
9,1010,204,David,Premium


In [10]:
cus_summary=df_merged.groupby(["CustomerName","CustomerID"]).agg(
    total_rev=("Revenue","sum"),
    num_ord=("OrderID","count"),
    mean_ord=("Revenue","mean")
)
cus_summary.sort_values("total_rev", ascending=False)

,,total_rev,num_ord,mean_ord
CustomerName,CustomerID,,,
Alice,201,4800,3,1600.0
David,204,1000,2,500.0
Eva,205,900,2,450.0
Carol,203,250,2,125.0
Bob,202,180,2,90.0
Frank,206,100,1,100.0


In [18]:
new_orders = pd.DataFrame({
    "OrderID": [1013, 1014, 1015],
    "CustomerID": [201, 207, 208],
    "Revenue": [500, 700, 300]
})
print(customers)

   CustomerID CustomerName CustomerType
0         201        Alice      Regular
1         202          Bob      Premium
2         203        Carol      Regular
3         204        David      Premium
4         205          Eva      Regular
5         206        Frank      Premium


In [15]:
leftmerge=pd.merge(new_orders,customers,on="CustomerID",how="left")
print(leftmerge)

   OrderID  CustomerID  Revenue CustomerName CustomerType
0     1013         201      500        Alice      Regular
1     1014         207      700          NaN          NaN
2     1015         208      300          NaN          NaN


In [16]:
innermerge=pd.merge(new_orders,customers,on="CustomerID",how="inner")
print(innermerge)

   OrderID  CustomerID  Revenue CustomerName CustomerType
0     1013         201      500        Alice      Regular


In [17]:
outermerge=pd.merge(new_orders,customers,on="CustomerID",how="outer")
print(outermerge)

   OrderID  CustomerID  Revenue CustomerName CustomerType
0   1013.0         201    500.0        Alice      Regular
1      NaN         202      NaN          Bob      Premium
2      NaN         203      NaN        Carol      Regular
3      NaN         204      NaN        David      Premium
4      NaN         205      NaN          Eva      Regular
5      NaN         206      NaN        Frank      Premium
6   1014.0         207    700.0          NaN          NaN
7   1015.0         208    300.0          NaN          NaN


In [20]:
leftmerge.isna().sum()

OrderID         0
CustomerID      0
Revenue         0
CustomerName    2
CustomerType    2
dtype: int64

In [21]:
leftmerge.loc[leftmerge["CustomerName"].isna()]

,OrderID,CustomerID,Revenue,CustomerName,CustomerType
1,1014,207,700,NaN,NaN
2,1015,208,300,NaN,NaN


In [22]:
leftmerge.loc[
    leftmerge["CustomerName"].isna(),
    ["OrderID", "CustomerID"]
]

,OrderID,CustomerID
1,1014,207
2,1015,208
